# 02 · Training a Behavior Cloning Policy
**Time: 50–60 minutes, including discussion and both real training/evaluation runs.**

## Learning Objectives
Explain behavior cloning (BC), interpret training and validation loss, and conduct a controlled learning-rate experiment.

## Background
BC predicts the expert action from the current observation. The optimizer updates network weights to reduce mean squared action error (MSE). Validation uses unseen episodes to select a checkpoint (saved model weights); test rollouts never update the model or select the primary variant. A small action error on recorded states does not guarantee successful closed-loop behavior.

## Step-by-step Instructions
### Step 1 — Inspect the frozen experiment
Keep the dataset, episode split, architecture (three 256-unit hidden layers), initialization seed 17, batch size and 50-epoch budget fixed. Compare learning rates **0.001 and 0.0003**. This is mandatory. Both variants start with identical weights. `lr_a` is the primary model, chosen before seeing test results.


In [ ]:
from pathlib import Path
import os, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
from manipulation_lab.teaching import read_json, markdown_table
GROUP = os.environ.get("LAB_GROUP", "group_01")  # Use the group assigned by your instructor.
RUN = workspace(GROUP)
settings = read_json(ROOT / "configs/bc_student.json")
settings["device"] = os.environ.get("LAB_DEVICE", settings["device"])
design = read_json(ROOT / "configs/controlled_experiment.json")
display(Markdown(markdown_table([settings])))
print("Controlled experiment:", design)


### Step 2 — Inspect the existing model and expert labels
The code below shows the unchanged robomimic model. Its training step predicts actions, computes MSE against expert actions, differentiates that loss, and updates weights. An epoch is one complete pass over the training examples.


In [ ]:
import h5py, torch
from manipulation_lab.policy import make_config, make_model
from manipulation_lab.dataset import inspect
info = inspect(DATA)
example_model = make_model(make_config(settings), info["policy_observation_shapes"], torch.device("cpu"))
print(example_model.nets["policy"])
with h5py.File(DATA, "r") as f:
    episode = info["splits"]["train"][0]
    print("First three expert action labels:", f[f"data/{episode}/actions"][:3])
del example_model


### Step 3 — Run both configurations
The next cell performs **two actual training jobs**, reloads both selected checkpoints and executes the same standard 5 + shifted 5 cases for each model. Jobs wait for the shared server resource slots. Keep this cell running until it finishes.

For a CPU installation, set `LAB_DEVICE=cpu` before starting Jupyter (see README). CUDA is not silently replaced by CPU. Do not change the seed, architecture or budget. Rerunning this cell creates another preserved experiment; do not rerun to select a higher test score.


In [ ]:
EXPERIMENT = next_stage(RUN, "experiment")
BASE_CONFIG = RUN / (EXPERIMENT.name + "_base_config.json")
write_json(BASE_CONFIG, settings)
command("controlled_experiment.py", "--config", BASE_CONFIG, "--dataset", DATA,
        "--design", ROOT / "configs/controlled_experiment.json",
        "--evaluation-config", ROOT / "configs/evaluation.yaml", "--output", EXPERIMENT, timeout=14400)
experiment = read_json(EXPERIMENT / "experiment.json")
assert experiment["complete"] and all(experiment["checks"].values())
print("Controlled checks:", experiment["checks"])
for variant, paths in experiment["variants"].items():
    print(variant, "checkpoint:", EXPERIMENT / paths["training"] / "student_policy.pth",
          "SHA256:", paths["checkpoint_sha256"])


### Step 4 — Compare convergence and saved checkpoints
The dashed line marks the lowest validation-loss epoch. Both models use the same training budget even if their selected epochs differ. The learning-rate comparison plot and CSV contain both variants, including unsuccessful rollouts.


In [ ]:
display(Image(filename=str(EXPERIMENT / "learning_rate_curves.png")))
display(Markdown(markdown_table(experiment["comparison"])))
for variant, paths in experiment["variants"].items():
    display(Markdown("### " + variant + " · learning rate " + str(paths["learning_rate"])))
    display(Image(filename=str(EXPERIMENT / paths["training"] / "training_curve.png")))
print("Both configurations, loss CSVs, checkpoints and 20 simulated episodes:", EXPERIMENT)


## Expected Outputs
Two 50-epoch loss histories, two selected checkpoints, the initial-weight equality check, and the same 10 test conditions for each checkpoint. A completed run need not have a high success rate.

## Questions for Students
**Q2. How does behavior cloning learn from demonstrations?** Identify input, target, loss and parameter update.

**Q3. Does a lower training loss always result in better manipulation performance?** Compare both curves and actual success counts. Explain what five cases per condition cannot establish. You can revise this answer after Notebook 03.


In [ ]:
Q2 = ""
Q3 = ""
answer_path = RUN / "discussion_answers.json"
answers = read_json(answer_path) if answer_path.exists() else {}
answers.update(q2=Q2, q3=Q3)
write_json(answer_path, answers)


## Completion Checklist
- [ ] Both configurations completed with only learning rate changed.
- [ ] I located both checkpoints, loss CSVs, selected epochs and configurations.
- [ ] I compared convergence and test performance without changing seeds.
- [ ] I saved Q2 and Q3. Continue to Notebook 03 to inspect the same evaluation evidence.
